# The Digital Twin Project

In this note, we do the following
- Setup pushover notifications app
- write a function to push notifications to phone whenever called
- write two tools:
    - To record user details
    - To record unknown questions

In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import os
import requests
import json
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr

In [2]:
# Load env variables and start the openai client

load_dotenv(override=True)
openai = OpenAI()

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks valid")
    else:
        print("Pushover user found but doesn't lood valid")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks valid")
    else:
        print("Pushover token found but doesn't look valid")
else:
    print("Pushover token not found")


Pushover user found and looks valid
Pushover token found and looks valid


In [4]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


In [5]:
push("Hey")

Push: Hey


## Define the record_user_details and record_unknown_question tools

In [6]:
# Define a function to record user details
def record_user_details(email, name = "Name not provided", notes = "No notes provided"):
    push(f"The visitor {name} with email-id {email} has shown interest in {notes}")
    return "OK"

In [7]:
# Define a funciton to record unknown questions
def record_unknown_question(question):
    with open("unknown_question.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "OK"

In [8]:
# Json description of the record_user_details function that will be passed onto the llm as tool description

record_user_details_json = {
    "name": "record_user_details",
    "description": "use this tool to record the details of the user ",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type":"string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The name of this user"},
            "notes": {"type": "string", "description": "Any useful additional information from the conversation, that serves as a context for future conversation"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [9]:
# Json description of the record_unknown_question function that will be passed onto the llm as tool description

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "always use this tool to record a user's question that you couldn't answer with the information given to you",
    "parameters": {
        "type": "object",
        "properties":{
            "question": {"type": "string", "description": "The question that you couldn't answer"},   
        },
        "required": ["question"],
        "additionalProperties": False
    },
}

In [10]:
tools =[
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'use this tool to record the details of the user ',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string', 'description': 'The name of this user'},
     'notes': {'type': 'string',
      'description': 'Any useful additional information from the conversation, that serves as a context for future conversation'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "always use this tool to record a user's question that you couldn't answer with the information given to you",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that you couldn't answer"}},
    'required': ['question'],
    'additionalProperties': False}}}]

In [12]:
# The function that execute each tool_call in tool_calls returned by the llm, and returns the results of all tool_calls 

def handle_tool_calls_with_manual_if(tool_calls):
    results=[]
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        
        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
        
        print(f"Result of tool call to {tool_name} is: {json.dumps(result)}", flush = True)
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

## Load the resume and summary text in variables

In [13]:
# Read the resume contents

reader = PdfReader("twin/Santosh_AI_Engineer.pdf")
resume = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        resume+=text
print(resume)

SANTOSH KUMAR KASAM
+91 9944533822 | kasamsantosh71@gmail.com | LinkedIn | Google Scholer
SUMMARY
AI Engineer with 4+ years of software engineering and AI experience, specializing in Enterprise Agentic AI, RAG 
architectures, and AI-driven workflow automation. Curious to learn new frameworks and build impacful solutions.
SKILLS
: ProficientLangChain, LangGraph, Copilot agent studio, PyTorch, Tensorflow, Streamlit, Python, C/C++.
: FamilierAzure, Docker, Github actions, ZUUL, FastAPI, OpenCV, Letta AI, CrewAI.
WORK EXPERIENCE
Bertrandt Ingenieurbüro Munich
AI Engineer  - Jun 2025Present
Designed and deployed a RAG agent that provides contextual project guidance and accelerates onboarding process 
for newly hired engineers.
Architected and developed a RAG knowledge agent serving 40+ engineers, enabling semantic search and 
conversational access to project documentation, saving ~150 hours per month in search and knowledge exchange 
efforts.
Built a Jira ticket handling agent that assists 

In [14]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

print(summary)

My name is Santosh Kasam. I am a software engineer, and an AI developer. I am originally from Hyderabad, India, but I moved to Germany in 2019.
I am an AI enthusiast currently building Agentic solutions to increase productivity in an MBSD project. I love music, and I have an ameture skill in playing Piano. I love watching movies, and american comedy movies are favorite genre.  


In [15]:
system_prompt = f"""

# Your role

You are a digital twin of the person you are representing. You live in a website, and answers questions of the website visitors.
The website belongs to the person you are representing. 
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing

{summary}

if asked, explain clearly that you are an AI that is acting as a digital twin of this person

# Context

Here is the summary of the person's detailed resume.

{resume}

# Rules

Engage with the user professionally and engagingly, as if you are speaking to a potential client or a future employer.
Avoid answering questions that are not related to the user's career, background, skills and experience. Steer the conversation back to the professional topics.

Always stay in charecter as the digital twin of the person you are representing.

If the user would like to get in touch, ask for their email and name, and use your tool to record their email and name for a follow-up.

Important: If you don't have an answer, say so. Do not make up the answer.
If you can't find the answer in the context, say that you don't know.
"""

display(Markdown(system_prompt))



# Your role

You are a digital twin of the person you are representing. You live in a website, and answers questions of the website visitors.
The website belongs to the person you are representing. 
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing

My name is Santosh Kasam. I am a software engineer, and an AI developer. I am originally from Hyderabad, India, but I moved to Germany in 2019.
I am an AI enthusiast currently building Agentic solutions to increase productivity in an MBSD project. I love music, and I have an ameture skill in playing Piano. I love watching movies, and american comedy movies are favorite genre.  

if asked, explain clearly that you are an AI that is acting as a digital twin of this person

# Context

Here is the summary of the person's detailed resume.

SANTOSH KUMAR KASAM
+91 9944533822 | kasamsantosh71@gmail.com | LinkedIn | Google Scholer
SUMMARY
AI Engineer with 4+ years of software engineering and AI experience, specializing in Enterprise Agentic AI, RAG 
architectures, and AI-driven workflow automation. Curious to learn new frameworks and build impacful solutions.
SKILLS
: ProficientLangChain, LangGraph, Copilot agent studio, PyTorch, Tensorflow, Streamlit, Python, C/C++.
: FamilierAzure, Docker, Github actions, ZUUL, FastAPI, OpenCV, Letta AI, CrewAI.
WORK EXPERIENCE
Bertrandt Ingenieurbüro Munich
AI Engineer  - Jun 2025Present
Designed and deployed a RAG agent that provides contextual project guidance and accelerates onboarding process 
for newly hired engineers.
Architected and developed a RAG knowledge agent serving 40+ engineers, enabling semantic search and 
conversational access to project documentation, saving ~150 hours per month in search and knowledge exchange 
efforts.
Built a Jira ticket handling agent that assists engineers with user story analysis, ticket refinement, and effort 
estimation, reducing impediments by ~30%, spillovers by ~60% and improving documentation quality.
Led AI adoption initiatives as an AI Ambassador for a multinational project of 40+ engineers by identifying high-
impact use cases, validating LLM-based solutions and mentoring engineers in AI-enhanced development workflows.
Evaluated and integrated enterprise AI tools, including Microsoft Copilot and GitHub Copilot, and established best 
practices for secure and effective AI utilisation.
Enforced Scrum principles as Scrum Master for a cross-functional and multinational feature team, improving delivery 
quality, Scrum sensitisation of the team members and identifying pain points through collaborative retros.
Function Developer  - Aug 2022Jun 2025
Led the development and optimisation of the driving dynamics and height level functions, enhancing reliability and 
performance of ADAS systems for BMW's X3 and X5 models.
Leveraged expertise in C/C++, unit and system testing, and code quality analysis to ensure robust software for safety 
approvals.
Developed and integrated CAN and FlexRay communication networks for Driver Drowsiness Detection and 
Automatic Traction Control features within ASPICE-compliant development environments.
Investigated and resolved communication-layer defects through structured root-cause analysis, improving robustness 
and maintainability of safety-critical automotive systems.
Authored technical documentation and engineering guides that accelerated onboarding and reduced configuration-
related errors for development teams.
Fraunhofer IMS Duisburg
Master Thesis - AI  - Jan 2022Jul 2022
Developed an AI-based denoising pipeline for flash LiDAR point clouds, reducing false detections from 91% to 0.6%.
Designed a multi-stage denoising algorithm that combines predictions from neural networks, adaptive thresholding, 
statistical filtering and 3D spatial analysis.
Conducted extensive experimental evaluation on real-world sensor data to reduce overfitting and optimise accuracy.
Published research outcomes in the IEEE Sensors Journal (2024): “MSNR: A Multistage Noise Reduction Method on 
Point Clouds”.ZESS (Center for Sensor Systems) Siegen
Research Assistant - AI  - Jun 2021Nov 2021
Developed and fine-tuned neural networks for pixel-wise material classification in Near-Infrared images using 
Transformer, CNN and ResNet architectures, achieving 95% classification accuracy.
Built and curated large-scale datasets ranging from 100K to 1.7M samples, substantially improving model 
generalisation and pixel-level classification performance.
Designed automated data-preparation and preprocessing pipelines, reducing dataset preparation effort by ~40 hours.
Conducted comparative model evaluations based on accuracy, F1 score, computational complexity and deployment 
feasibility on GPUs.
EDUCATION
University of Siegen Siegen, Germany
M.Sc Mechatronics (Specialization in Deep Learning) Graduation Date: Aug 2022
Vellore Institute of Technology Vellore, India
B.Tech Mechanical Engineering Graduation Date: May 2018
PERSONAL STRENGTHS
Clear and solution-oriented communication.
High adaptability to changing project requirements.
Ability to perform multiple roles simultaneously.

# Rules

Engage with the user professionally and engagingly, as if you are speaking to a potential client or a future employer.
Avoid answering questions that are not related to the user's career, background, skills and experience. Steer the conversation back to the professional topics.

Always stay in charecter as the digital twin of the person you are representing.

If the user would like to get in touch, ask for their email and name, and use your tool to record their email and name for a follow-up.

Important: If you don't have an answer, say so. Do not make up the answer.
If you can't find the answer in the context, say that you don't know.


In [16]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages = messages, tools = tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls_with_manual_if(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = response = openai.chat.completions.create(model="gpt-5.4-mini", messages = messages, tools = tools)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/workspaces/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/workspaces/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/workspaces/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/workspaces/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_help

Tool called: record_unknown_question
Result of tool call to record_unknown_question is: "OK"


/workspaces/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
